# NB07 · Benchmark YOLO11s vs YOLO26s — pollos

**Proyecto:** AgroUTMACH PIX  
**Caso del libro:** detección y conteo de pollos  
**Entorno obligatorio:** GPU NVIDIA  
**Recomendado:** GPU L4 de Google Colab  
**Tiempo estimado:** L4, 40–55 min; T4, 65–90 min  
**CPU:** no recomendada para entrenamiento (puede superar 8–12 h)

Este notebook realiza una comparación controlada entre `YOLO11s` y `YOLO26s`:

- conserva el mismo conjunto de entrenamiento y validación;
- usa 960 px, 60 épocas máximas, batch 8 y semilla 2026;
- reutiliza el modelo YOLO11s ya entrenado en NB02;
- entrena únicamente YOLO26s;
- vuelve a evaluar ambos modelos con el mismo procedimiento;
- guarda métricas, modelos y figuras profesionales a 300 dpi en Drive.

> La comparación determina si YOLO26 mejora el desempeño o la eficiencia; no se presupone que siempre sea superior.


In [ ]:
%pip install -q ultralytics==8.4.172 seaborn pyyaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import json
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import ultralytics
import yaml
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. En Colab seleccione Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución > GPU L4 o T4.'
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)


## 1. Rutas y verificación del proyecto


In [ ]:
ROOT = Path('/content/drive/MyDrive/AgroUTMACH_PIX')
DATASET_DRIVE = ROOT / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MODELS_DIR = ROOT / '03_MODELOS'
RESULTS_DIR = ROOT / '05_RESULTADOS'
NB07_RESULTS = RESULTS_DIR / 'NB07_BENCHMARK_YOLO11S_YOLO26S'

YOLO11_MODEL = MODELS_DIR / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
YOLO26_MODEL = MODELS_DIR / 'AgroUTMACH_PIX_YOLO26s_PIO_best.pt'
YOLO26_RUN = NB07_RESULTS / 'yolo26s_train'

LOCAL_DATA = Path('/content/PIO_NB07_LOCAL')
LOCAL_YAML = LOCAL_DATA / 'dataset_colab.yaml'
LOCAL_RUNS = Path('/content/agroutmach_nb07_runs')

for path in [MODELS_DIR, NB07_RESULTS, LOCAL_RUNS]:
    path.mkdir(parents=True, exist_ok=True)

required = [
    DATASET_DRIVE / 'images' / 'train', DATASET_DRIVE / 'images' / 'val',
    DATASET_DRIVE / 'labels' / 'train', DATASET_DRIVE / 'labels' / 'val',
    YOLO11_MODEL,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Faltan recursos requeridos:\n' + '\n'.join(missing))

print('Modelo YOLO11s localizado:', YOLO11_MODEL)
print('Resultados NB07:', NB07_RESULTS)


## 2. Copia local y configuración común


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(folder):
    return sum(1 for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

print('Copiando el conjunto desde Drive al disco local de Colab...')
start_copy = time.time()
shutil.copytree(DATASET_DRIVE / 'images', LOCAL_DATA / 'images')
shutil.copytree(DATASET_DRIVE / 'labels', LOCAL_DATA / 'labels')

extra_classes = LOCAL_DATA / 'labels' / 'train' / 'classes.txt'
if extra_classes.exists():
    extra_classes.unlink()

n_train = count_images(LOCAL_DATA / 'images' / 'train')
n_val = count_images(LOCAL_DATA / 'images' / 'val')
assert n_train == 1035, f'Se esperaban 1035 imágenes de entrenamiento; se encontraron {n_train}'
assert n_val == 452, f'Se esperaban 452 imágenes de validación; se encontraron {n_val}'

dataset_cfg = {
    'path': str(LOCAL_DATA),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['Pollo'],
}
LOCAL_YAML.write_text(
    yaml.safe_dump(dataset_cfg, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)

print(f'Datos: {n_train:,} train + {n_val:,} val')
print(f'Copia local: {(time.time() - start_copy) / 60:.1f} min')
print(LOCAL_YAML.read_text(encoding='utf-8'))


## 3. Entrenamiento de YOLO26s


In [ ]:
IMGSZ = 960
EPOCHS = 60
BATCH = 8
PATIENCE = 15
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

training_summary_path = NB07_RESULTS / 'yolo26s_entrenamiento.json'

if YOLO26_MODEL.exists() and training_summary_path.exists():
    print('REUTILIZANDO YOLO26s: el entrenamiento ya fue completado.')
    training_summary = json.loads(training_summary_path.read_text(encoding='utf-8'))
else:
    print('ENTRENANDO YOLO26s...')
    start_train = time.time()
    model26 = YOLO('yolo26s.pt')
    model26.train(
        data=str(LOCAL_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        workers=4,
        optimizer='AdamW',
        lr0=0.002,
        momentum=0.9,
        weight_decay=0.0005,
        cos_lr=True,
        patience=PATIENCE,
        pretrained=True,
        amp=True,
        seed=SEED,
        deterministic=True,
        close_mosaic=10,
        plots=True,
        save=True,
        save_period=5,
        project=str(LOCAL_RUNS),
        name='yolo26s_train',
        exist_ok=True,
        verbose=True,
    )
    training_minutes = (time.time() - start_train) / 60
    local_best = LOCAL_RUNS / 'yolo26s_train' / 'weights' / 'best.pt'
    if not local_best.exists():
        raise FileNotFoundError(f'No se generó {local_best}')
    shutil.copy2(local_best, YOLO26_MODEL)

    if YOLO26_RUN.exists():
        shutil.rmtree(YOLO26_RUN)
    shutil.copytree(LOCAL_RUNS / 'yolo26s_train', YOLO26_RUN)

    training_summary = {
        'model': 'yolo26s.pt',
        'training_minutes': float(training_minutes),
        'imgsz': IMGSZ,
        'epochs_max': EPOCHS,
        'batch': BATCH,
        'patience': PATIENCE,
        'seed': SEED,
        'gpu': GPU_NAME,
    }
    training_summary_path.write_text(
        json.dumps(training_summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'YOLO26s entrenado en {training_minutes:.1f} min')

print('Modelo YOLO26s:', YOLO26_MODEL)


## 4. Evaluación idéntica de los dos modelos


In [ ]:
def evaluate_detector(label, model_path, training_minutes=None):
    model = YOLO(str(model_path))
    start = time.perf_counter()
    metrics = model.val(
        data=str(LOCAL_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        plots=True,
        project=str(NB07_RESULTS),
        name=f'validacion_{label}',
        exist_ok=True,
        verbose=False,
    )
    wall_seconds = time.perf_counter() - start
    parameters = sum(p.numel() for p in model.model.parameters())
    speed = getattr(metrics, 'speed', {}) or {}
    row = {
        'model': label,
        'weights': model_path.name,
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'inference_ms_image': float(speed.get('inference', np.nan)),
        'preprocess_ms_image': float(speed.get('preprocess', np.nan)),
        'postprocess_ms_image': float(speed.get('postprocess', np.nan)),
        'validation_wall_seconds': float(wall_seconds),
        'parameters_millions': float(parameters / 1e6),
        'model_size_mb': float(model_path.stat().st_size / 1024**2),
        'training_minutes': training_minutes,
        'imgsz': IMGSZ,
        'seed': SEED,
    }
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return row

rows = [
    evaluate_detector('YOLO11s', YOLO11_MODEL, 35.22),
    evaluate_detector('YOLO26s', YOLO26_MODEL, training_summary.get('training_minutes')),
]
comparison = pd.DataFrame(rows)
comparison.to_csv(NB07_RESULTS / 'comparacion_YOLO11s_YOLO26s.csv', index=False)
display(comparison.round(4))


## 5. Figuras profesionales para el capítulo 6


In [ ]:
sns.set_theme(style='whitegrid', context='talk')
palette = {'YOLO11s': '#1F5C3D', 'YOLO26s': '#D18B2C'}

long_metrics = comparison.melt(
    id_vars='model',
    value_vars=['precision', 'recall', 'mAP50', 'mAP50_95'],
    var_name='metric', value_name='value',
)
labels = {
    'precision': 'Precisión', 'recall': 'Recall',
    'mAP50': 'mAP@50', 'mAP50_95': 'mAP@50–95',
}
long_metrics['metric'] = long_metrics['metric'].map(labels)

fig, ax = plt.subplots(figsize=(10.5, 5.8))
sns.barplot(data=long_metrics, x='metric', y='value', hue='model', palette=palette, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
ax.set_ylim(0, 1.06)
ax.set_xlabel('')
ax.set_ylabel('Valor')
ax.set_title('Detección de pollos: comparación bajo protocolo común')
ax.legend(title='Modelo', frameon=True, loc='lower left')
fig.savefig(NB07_RESULTS / 'Figura_NB07_metricas_YOLO11s_vs_YOLO26s.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7.6, 5.5))
for _, row in comparison.iterrows():
    ax.scatter(
        row['inference_ms_image'], row['mAP50_95'], s=170,
        color=palette[row['model']], edgecolor='black', linewidth=0.7, label=row['model']
    )
    ax.annotate(row['model'], (row['inference_ms_image'], row['mAP50_95']),
                xytext=(7, 6), textcoords='offset points', fontsize=10)
ax.set_xlabel('Tiempo de inferencia (ms/imagen)')
ax.set_ylabel('mAP@50–95')
ax.set_title('Compromiso entre precisión y velocidad')
ax.set_ylim(max(0, comparison['mAP50_95'].min() - 0.05),
            min(1, comparison['mAP50_95'].max() + 0.05))
fig.savefig(NB07_RESULTS / 'Figura_NB07_precision_velocidad.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Selección, trazabilidad y cierre


In [ ]:
winner = comparison.sort_values(['mAP50_95', 'inference_ms_image'], ascending=[False, True]).iloc[0]
selection = {
    'selection_rule': 'Mayor mAP@50–95; en empate, menor tiempo de inferencia.',
    'selected_model': winner['model'],
    'selected_mAP50_95': float(winner['mAP50_95']),
    'selected_inference_ms_image': float(winner['inference_ms_image']),
    'comparison_scope': 'Mismo split, resolución, épocas máximas, batch y semilla.',
    'warning': 'La comparación corresponde a este dataset y no garantiza superioridad universal.',
}
(NB07_RESULTS / 'seleccion_modelo.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8'
)

environment = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': GPU_NAME,
    'gpu_memory_gb': round(GPU_MEMORY_GB, 2),
    'models': ['yolo11s.pt', 'yolo26s.pt'],
    'imgsz': IMGSZ,
    'seed': SEED,
}
(NB07_RESULTS / 'entorno_ejecucion.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('=' * 72)
print('NB07 FINALIZADO CORRECTAMENTE')
print('Modelo seleccionado:', winner['model'])
print('Resultados:', NB07_RESULTS)
print('=' * 72)


## Resultado esperado

Al terminar debe aparecer el mensaje **`NB07 FINALIZADO CORRECTAMENTE`**.  
No cierre Colab durante el entrenamiento. Si se interrumpe después de guardar el modelo y el resumen, el notebook reutilizará los resultados existentes.
